In [1]:
from google.colab import files

uploaded = files.upload()

Saving gender_submission.csv to gender_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


In [2]:
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [3]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
train_df.shape

(891, 12)

In [5]:
train_df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [6]:
train_df["Survived"].value_counts()

,count
Survived,
0,549
1,342


In [7]:
train_df = train_df.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis =1)

In [8]:
train_df["Embarked"].value_counts()

,count
Embarked,
S,644
C,168
Q,77


In [9]:
train_df["Age"].value_counts()

,count
Age,
24.00,30
22.00,27
18.00,26
28.00,25
30.00,25
...,...
24.50,1
0.67,1
0.42,1


In [10]:
train_df["Age"].describe()

,Age
count,714.000000
mean,29.699118
std,14.526497
min,0.420000
25%,20.125000
50%,28.000000
75%,38.000000
max,80.000000


In [11]:
train_df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='object')

In [12]:
train_df["Age"] = train_df["Age"].fillna(28)

In [13]:
train_df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='object')

In [14]:
train_df["Embarked"] = train_df["Embarked"].fillna("S")

In [15]:
train_df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


In [16]:
train_df["Sex"] = train_df["Sex"].replace(
    {
        "male": 0,
        "female": 1
    }
)

/tmp/ipykernel_4542/3127983431.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df["Sex"] = train_df["Sex"].replace(


In [17]:
train_df["Sex"].unique()

array([0, 1])

In [18]:
dummy_df = pd.get_dummies(train_df["Embarked"])

In [19]:
train_df = pd.concat([train_df, dummy_df], axis =1)

In [20]:
train_df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'C', 'Q', 'S'],
      dtype='object')

In [21]:
train_df = train_df.drop(["Embarked"], axis=1)

In [22]:
train_df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'C', 'Q',
       'S'],
      dtype='object')

In [23]:
train_df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'C', 'Q',
       'S'],
      dtype='object')

In [24]:
y = train_df["Survived"]
X = train_df.drop(["Survived"], axis = 1)

In [25]:
from sklearn.model_selection import train_test_split


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2
)

In [27]:
X_train.shape

(712, 9)

In [28]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [29]:
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [30]:
type(X_train)

numpy.ndarray

In [31]:
y_train.shape

(712,)

In [32]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)

In [33]:
from torch.utils.data import TensorDataset, DataLoader

In [34]:
train_dataset = TensorDataset(X_train, y_train)

In [35]:
len(train_dataset)

712

In [36]:
train_loader = DataLoader(
    train_dataset,
    batch_size = 64,
    shuffle = True
)

In [37]:
for features, labels in train_loader:
  print(features.shape)
  print(labels.shape)
  break

torch.Size([64, 9])
torch.Size([64])


In [38]:
import torch.nn as nn

class Titanic(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(9,16)
    self.layer2 = nn.Linear(16,8)
    self.layer3 = nn.Linear(8,1)

    self.relu = nn.ReLU()
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):

    x = self.layer1(x)

    x = self.relu(x)

    x = self.layer2(x)

    x = self.relu(x)

    x = self.layer3(x)

    x = self.sigmoid(x)

    return x

In [39]:
model = Titanic()
print(model)

Titanic(
  (layer1): Linear(in_features=9, out_features=16, bias=True)
  (layer2): Linear(in_features=16, out_features=8, bias=True)
  (layer3): Linear(in_features=8, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [40]:
criterion = nn.BCELoss()

In [41]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.01
)

In [42]:
outputs = model(features)

In [43]:
print(outputs.shape)
print(labels.shape)

torch.Size([64, 1])
torch.Size([64])


In [44]:
labels = labels.unsqueeze(1)

In [45]:
print(labels.shape)

torch.Size([64, 1])


In [54]:
for epoch in range(50):
  for features, labels in train_loader:
    labels = labels.unsqueeze(1)

    outputs = model(features)

    loss = criterion(outputs, labels)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

  if epoch % 10 == 0:
    print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

Epoch 0: Loss = 0.2073
Epoch 10: Loss = 0.2984
Epoch 20: Loss = 0.4694
Epoch 30: Loss = 0.1781
Epoch 40: Loss = 0.3233


In [55]:
model.eval()

Titanic(
  (layer1): Linear(in_features=9, out_features=16, bias=True)
  (layer2): Linear(in_features=16, out_features=8, bias=True)
  (layer3): Linear(in_features=8, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)

In [56]:
outputs = model(X_test)

In [57]:
outputs.shape

torch.Size([179, 1])

In [58]:
predictions = outputs > 0.5

In [60]:
print(predictions.shape)
print(y_test.shape)

torch.Size([179, 1])
torch.Size([179])


In [61]:
predictions = predictions.squeeze()

In [62]:
(predictions == y_test).float().mean()

tensor(0.8156)

In [65]:
model.eval()

outputs = model(X_train)
predictions = outputs > 0.5

predictions = predictions.squeeze()

(predictions == y_train).float().mean()

tensor(0.8680)